## SelfGNN — Finance Merchant Dataset Preprocessing

**Input:** `datasetRaw/finance/transactions_data.csv` (13.3M rows, ~2K users, ~6K merchants)

| File | Description |
|------|-------------|
| `trn_mat_time` | `[global_mat, subMat (T=10), timeMat]` pickle |
| `sequence` | Per-user chronological merchant visit list |
| `tst_int` / `val_int` | Held-out positive merchant per user |
| `test_dict` / `val_dict` | `{uid → [neg_merchants]}` (999 each) |
| `user2id.pkl` / `merchant2id.pkl` | ID mappings for feature extraction |
| `train/test/val_finance_merchant.csv` | TSV exports |
| `edge_weights.pkl` | `{(uid,mid) → avg_abs_amount}` for `--use_edge_features` |

**Configuration:** `MIN_INTERACTIONS=5` (5-core), `GRAPH_NUM=10` annual slices (2010–2019), `TEST_NEG=min(999, merchantnum-1)`

In [1]:
# ── Configuration ──────────────────────────────────────────────────────────────
OUT_DIR          = '../Datasets/finance-merchant/'
RAW_TXN          = '../datasetRaw/finance/transactions_data.csv'
MIN_INTERACTIONS = 5
GRAPH_NUM        = 8        # 8 time-interval sub-graphs
LAST_N_YEARS     = 2        # keep only the most recent 2 years of events
SECS_PER_YEAR    = 365 * 86400
TEST_PER_GROUP   = 3_333    # 3 terciles x 3333 balanced test/val users
SEED             = 42       # aligned with all training runs

# ── Imports ────────────────────────────────────────────────────────────────────
import os, pickle, copy, ast
from collections import defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

os.makedirs(OUT_DIR, exist_ok=True)
print('Libraries loaded.')
print('Output dir :', OUT_DIR)
print('Raw path   :', RAW_TXN)
print(f'Config     : GRAPH_NUM={GRAPH_NUM}, LAST_N_YEARS={LAST_N_YEARS}, '
      f'TEST_PER_GROUP={TEST_PER_GROUP}, SEED={SEED}')

Libraries loaded.
Output dir : ../Datasets/finance-merchant/
Raw path   : ../datasetRaw/finance/transactions_data.csv
Config     : GRAPH_NUM=8, LAST_N_YEARS=2, TEST_PER_GROUP=3333, SEED=42


In [2]:
# ── Cell 2: Load CSV chunks ─────────────────────────────────────────────────────
# Finance CSV uses dollar signs in amount column; strip them before parsing.

chunks = []
for chunk in pd.read_csv(RAW_TXN, chunksize=500_000, low_memory=False):
    chunk['amount'] = pd.to_numeric(
        chunk['amount'].astype(str).str.replace('$', '', regex=False),
        errors='coerce',
    )
    chunk = chunk.dropna(subset=['client_id', 'merchant_id', 'date', 'amount'])
    chunks.append(chunk[['client_id', 'merchant_id', 'date', 'amount']])

df = pd.concat(chunks, ignore_index=True)
df['date']        = pd.to_datetime(df['date'], errors='coerce')
df                = df.dropna(subset=['date'])
df['unix_ts']     = df['date'].astype(np.int64) // 10**9
df['merchant_id'] = df['merchant_id'].astype(str)
df['client_id']   = df['client_id'].astype(str)

print(f'Loaded     : {len(df):,} transactions')
print(f'Date range : {df["date"].min()} → {df["date"].max()}')
print(f'Users      : {df["client_id"].nunique():,}')
print(f'Merchants  : {df["merchant_id"].nunique():,}')

# ── Last-N-years filter ────────────────────────────────────────────────────────
_all_ts_max = int(df['unix_ts'].max())
_cutoff     = _all_ts_max - LAST_N_YEARS * SECS_PER_YEAR
df          = df[df['unix_ts'] >= _cutoff].reset_index(drop=True)
print(f'\nLast-{LAST_N_YEARS}yr filter:')
print(f'  cutoff       : {datetime.fromtimestamp(_cutoff)}')
print(f'  max          : {datetime.fromtimestamp(_all_ts_max)}')
print(f'  kept txns    : {len(df):,}')
print(f'  kept users   : {df["client_id"].nunique():,}')
print(f'  kept mrchnts : {df["merchant_id"].nunique():,}')

Loaded     : 13,305,915 transactions
Date range : 2010-01-01 00:01:00 → 2019-10-31 23:59:00
Users      : 1,219
Merchants  : 74,831

Last-2yr filter:
  cutoff       : 2017-11-01 07:59:00
  max          : 2019-11-01 07:59:00
  kept txns    : 2,789,098
  kept users   : 1,210
  kept mrchnts : 38,946


In [3]:
# ── Cell 3: K-core filter ───────────────────────────────────────────────────────
def kcore_filter_df(df: pd.DataFrame, user_col: str, item_col: str, k: int = MIN_INTERACTIONS) -> pd.DataFrame:
    for iteration in range(1000):
        u_deg = df[user_col].value_counts()
        i_deg = df[item_col].value_counts()
        valid_u = set(u_deg[u_deg >= k].index)
        valid_i = set(i_deg[i_deg >= k].index)
        new_df = df[df[user_col].isin(valid_u) & df[item_col].isin(valid_i)]
        print(f'  Iter {iteration+1}: {new_df[user_col].nunique():,} users, '
              f'{new_df[item_col].nunique():,} merchants, {len(new_df):,} edges')
        if len(new_df) == len(df):
            break
        df = new_df.reset_index(drop=True)
    return df

print(f'K-core filtering (k={MIN_INTERACTIONS})...')
df_filt = kcore_filter_df(df, 'client_id', 'merchant_id')
print(f'After k-core: {df_filt["client_id"].nunique():,} users, '
      f'{df_filt["merchant_id"].nunique():,} merchants, {len(df_filt):,} transactions')

K-core filtering (k=5)...
  Iter 1: 1,210 users, 17,377 merchants, 2,749,810 edges
  Iter 2: 1,210 users, 17,377 merchants, 2,749,810 edges
After k-core: 1,210 users, 17,377 merchants, 2,749,810 transactions


In [4]:
# ── Cell 4: ID mapping + interaction dict + minn/maxx + edge_weights ───────────
user_strs     = sorted(df_filt['client_id'].unique())
merchant_strs = sorted(df_filt['merchant_id'].unique())

user2id     = {u: i for i, u in enumerate(user_strs)}
merchant2id = {m: i for i, m in enumerate(merchant_strs)}

usrnum      = len(user2id)
merchantnum = len(merchant2id)
TEST_NEG    = min(999, merchantnum - 1)

df_filt = df_filt.copy()
df_filt['uid'] = df_filt['client_id'].map(user2id)
df_filt['mid'] = df_filt['merchant_id'].map(merchant2id)

with open(OUT_DIR + 'user2id.pkl', 'wb') as f:
    pickle.dump(user2id, f)
with open(OUT_DIR + 'merchant2id.pkl', 'wb') as f:
    pickle.dump(merchant2id, f)
print(f'Saved: user2id.pkl ({usrnum:,}), merchant2id.pkl ({merchantnum:,})')

# Build interaction dict
interaction: list = [defaultdict(list) for _ in range(usrnum)]
for row in df_filt[['uid', 'mid', 'unix_ts']].itertuples(index=False):
    interaction[row.uid][row.mid].append(row.unix_ts)
for uid in range(usrnum):
    for mid in interaction[uid]:
        interaction[uid][mid] = sorted(interaction[uid][mid])

# Anchor to the LAST_N_YEARS window for deterministic 8-slice boundaries.
minn = _cutoff
maxx = max(_all_ts_max, int(df_filt['unix_ts'].max()))

n_pairs  = df_filt.groupby(['uid', 'mid']).ngroups
n_events = len(df_filt)
density  = n_pairs / (usrnum * merchantnum) * 100

print(f'usrnum      = {usrnum:,}')
print(f'merchantnum = {merchantnum:,}')
print(f'Unique pairs: {n_pairs:,} | Total events: {n_events:,}')
print(f'Density     : {density:.4f}%')
print(f'Time range  : {pd.Timestamp(minn, unit="s")} → {pd.Timestamp(maxx, unit="s")}')

# Edge weights: average |amount| per (uid, mid) pair
ew_sum: dict = defaultdict(float)
ew_cnt: dict = defaultdict(int)
for row in df_filt[['uid', 'mid', 'amount']].itertuples(index=False):
    k = (int(row.uid), int(row.mid))
    ew_sum[k] += abs(float(row.amount))
    ew_cnt[k] += 1
edge_weights = {k: ew_sum[k] / ew_cnt[k] for k in ew_cnt}
print(f'Edge weights: {len(edge_weights):,} pairs')

Saved: user2id.pkl (1,210), merchant2id.pkl (17,377)
usrnum      = 1,210
merchantnum = 17,377
Unique pairs: 136,459 | Total events: 2,749,810
Density     : 0.6490%
Time range  : 2017-10-31 23:59:00 → 2019-10-31 23:59:00
Edge weights: 136,459 pairs


In [5]:
# ── Cell 5: Dataset statistics ──────────────────────────────────────────────────
try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    user_degrees     = [len(interaction[u]) for u in range(usrnum) if interaction[u]]
    merchant_deg_cnt: dict = defaultdict(int)
    for u in range(usrnum):
        for mid in interaction[u]:
            merchant_deg_cnt[mid] += 1
    merchant_degrees = list(merchant_deg_cnt.values())

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle('Finance-Merchant Dataset Statistics', fontweight='bold')

    axes[0].hist(user_degrees, bins=50, color='#1976D2', alpha=0.8)
    axes[0].set_xlabel('Merchants per user')
    axes[0].set_ylabel('Count')
    axes[0].set_title('User degree distribution')
    axes[0].set_yscale('log')

    axes[1].hist(merchant_degrees, bins=50, color='#F57C00', alpha=0.8)
    axes[1].set_xlabel('Users per merchant')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Merchant degree distribution')
    axes[1].set_yscale('log')

    plt.tight_layout()
    plt.savefig(OUT_DIR + 'dataset_stats.png', bbox_inches='tight', dpi=120)
    plt.show()
    print('Saved: dataset_stats.png')
except ImportError:
    print('matplotlib not available, skipping plot')

print(f'Users         : {usrnum:>10,}')
print(f'Merchants     : {merchantnum:>10,}')
print(f'Unique pairs  : {n_pairs:>10,}')
print(f'Total events  : {n_events:>10,}')
print(f'Density       : {density:>10.4f}%')

Saved: dataset_stats.png
Users         :      1,210
Merchants     :     17,377
Unique pairs  :    136,459
Total events  :  2,749,810
Density       :     0.6490%


C:\Users\User\AppData\Local\Temp\ipykernel_15428\2439734565.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# ── Cell 6: Train / test / val split ───────────────────────────────────────────

def negSamp(pos_set: set, samp_size: int, node_num: int) -> list:
    valid = [j for j in range(node_num) if j not in pos_set]
    if not valid:
        valid = list(range(node_num))
    samp_size = min(samp_size, len(valid))
    idx = np.random.choice(len(valid), size=samp_size, replace=False)
    return [valid[i] for i in idx]


def split_test_val(
    interaction: list,
    usrnum: int,
    merchantnum: int,
    test_neg: int = TEST_NEG,
    per_group: int = TEST_PER_GROUP,
    seed: int = SEED,
) -> tuple:
    rng = np.random.default_rng(seed)
    np.random.seed(seed)

    # Tercile split by total event count so low/mid/high test segments are
    # directly comparable (addresses critic's point on unbalanced sampling).
    activity   = np.zeros(usrnum, dtype=np.int64)
    n_distinct = np.zeros(usrnum, dtype=np.int32)
    for u in range(usrnum):
        if interaction[u]:
            activity[u]   = sum(len(tss) for tss in interaction[u].values())
            n_distinct[u] = len(interaction[u])
    eligible = np.array(
        [u for u in range(usrnum) if activity[u] >= 3 and n_distinct[u] >= 2],
        dtype=np.int64,
    )
    elig_act = activity[eligible]
    t1, t2   = np.quantile(elig_act, [1/3, 2/3])
    pool_low  = eligible[elig_act <= t1]
    pool_mid  = eligible[(elig_act > t1) & (elig_act <= t2)]
    pool_high = eligible[elig_act > t2]
    print(f'Tercile thresholds : low<={t1:.0f}, mid<={t2:.0f}, high>{t2:.0f}')
    print(f'Pool sizes         : low={len(pool_low):,} mid={len(pool_mid):,} high={len(pool_high):,}')

    def _pick(pool):
        size = min(per_group, len(pool))
        return rng.choice(pool, size=size, replace=False) if size > 0 else np.array([], dtype=np.int64)

    pick_low, pick_mid, pick_high = _pick(pool_low), _pick(pool_mid), _pick(pool_high)
    pick_usr = np.concatenate([pick_low, pick_mid, pick_high])
    segments = {
        'low':  [int(u) for u in pick_low],
        'mid':  [int(u) for u in pick_mid],
        'high': [int(u) for u in pick_high],
    }
    print(f'Selected           : low={len(pick_low):,} mid={len(pick_mid):,} high={len(pick_high):,}')

    tst_int   = [None] * usrnum
    val_int   = [None] * usrnum
    test_rows: list = []
    val_rows:  list = []
    skipped = 0

    for u in pick_usr:
        merch = interaction[u]
        if not merch or len(merch) < 3:
            skipped += 1
            continue

        pairs = sorted((ts, mid) for mid, tss in merch.items() for ts in tss)

        tst_time, tst_mid = pairs[-1]
        val_mid = val_time = None
        for ts, mid in reversed(pairs[:-1]):
            if mid != tst_mid:
                val_mid, val_time = mid, ts
                break
        if val_mid is None:
            skipped += 1
            continue

        # Remove only the held-out timestamp
        tss = [t for t in interaction[u][tst_mid] if t != tst_time]
        if tss:
            interaction[u][tst_mid] = tss
        else:
            del interaction[u][tst_mid]

        tss = [t for t in interaction[u].get(val_mid, []) if t != val_time]
        if tss:
            interaction[u][val_mid] = tss
        elif val_mid in interaction[u]:
            del interaction[u][val_mid]

        if sum(1 for tss in interaction[u].values() if tss) < 1:
            skipped += 1
            continue

        tst_int[u] = tst_mid
        val_int[u] = val_mid
        all_pos    = set(merch.keys())

        test_rows.append({'user_id': u+1, 'merchant_id': tst_mid,
                          'time': tst_time, 'neg_merchants': str(negSamp(all_pos, test_neg, merchantnum))})
        val_rows.append( {'user_id': u+1, 'merchant_id': val_mid,
                          'time': val_time,  'neg_merchants': str(negSamp(all_pos, test_neg, merchantnum))})

    pd.DataFrame(test_rows).to_csv(OUT_DIR + 'test_finance_merchant.csv', sep='\t', index=False)
    pd.DataFrame(val_rows).to_csv( OUT_DIR + 'val_finance_merchant.csv',  sep='\t', index=False)

    n_test  = sum(1 for x in tst_int if x is not None)
    n_val   = sum(1 for x in val_int  if x is not None)
    per_seg = {k: sum(1 for u in v if tst_int[u] is not None) for k, v in segments.items()}
    print(f'Test users : {n_test:,}  (per segment: {per_seg})')
    print(f'Val  users : {n_val:,}')
    print(f'Skipped    : {skipped:,}')
    meta = {'thresholds': [float(t1), float(t2)],
            'criterion': 'total_event_count',
            'per_group_target': int(per_group),
            'per_segment_count': per_seg}
    return interaction, tst_int, val_int, segments, meta


interaction_train = copy.deepcopy(interaction)
interaction_train, tstInt, valInt, segments, segment_meta = split_test_val(interaction_train, usrnum, merchantnum)

Tercile thresholds : low<=1691, mid<=2454, high>2454
Pool sizes         : low=404 mid=403 high=403
Selected           : low=404 mid=403 high=403
Test users : 1,210  (per segment: {'low': 404, 'mid': 403, 'high': 403})
Val  users : 1,210
Skipped    : 0


In [ ]:
# ── Cell 7: Balanced tercile split → train / test / val ──────────────────────
# Chronological split (seed=SEED), but with balanced low/mid/high interaction
# segments so per-group test metrics are directly comparable (fixes critic's
# point that the previous 10K random sample could under-represent low-activity
# users). Steps:
#   1) Rank eligible users by total event count (activity level).
#   2) Tercile split on that rank → low / mid / high pools.
#   3) Sample TEST_PER_GROUP users from each tercile → total 9,999 test/val users.
#   4) For each picked user:
#      - Last interaction timestamp                → test positive
#      - Second-last on a *different* merchant    → val positive
#      - All remaining timestamps                 → training
# Only the specific held-out timestamp is removed; earlier visits to the same
# merchant stay in training. Independent 999-neg samples drawn for test & val.
# Segment membership is persisted to user_segments.pkl in Cell 9.

def negSamp(pos_set: set, samp_size: int, node_num: int) -> list[int]:
    """Sample `samp_size` negative merchant IDs not in `pos_set`."""
    valid = [j for j in range(node_num) if j not in pos_set]
    if not valid:
        valid = list(range(node_num))
    samp_size = min(samp_size, len(valid))
    idx = np.random.choice(len(valid), size=samp_size, replace=False)
    return [valid[i] for i in idx]


def split_test_val(
    interaction: list,
    usrnum: int,
    merchantnum: int,
    test_neg: int = TEST_NEG,
    per_group: int = TEST_PER_GROUP,
    seed: int = SEED,
) -> tuple:
    rng = np.random.default_rng(seed)
    np.random.seed(seed)

    # Eligibility: user must have ≥3 events AND ≥2 distinct merchants.
    activity = np.zeros(usrnum, dtype=np.int64)
    n_distinct = np.zeros(usrnum, dtype=np.int32)
    for u in range(usrnum):
        if interaction[u] is not None:
            activity[u]   = sum(len(tss) for tss in interaction[u].values())
            n_distinct[u] = len(interaction[u])

    eligible = np.array(
        [u for u in range(usrnum) if activity[u] >= 3 and n_distinct[u] >= 2],
        dtype=np.int64,
    )
    elig_act = activity[eligible]
    t1, t2   = np.quantile(elig_act, [1/3, 2/3])

    pool_low  = eligible[elig_act <= t1]
    pool_mid  = eligible[(elig_act > t1) & (elig_act <= t2)]
    pool_high = eligible[elig_act > t2]
    print(f'Tercile thresholds : low≤{t1:.0f}, mid≤{t2:.0f}, high>{t2:.0f}')
    print(f'Pool sizes         : low={len(pool_low):,}  mid={len(pool_mid):,}  high={len(pool_high):,}')

    def _pick(pool: np.ndarray) -> np.ndarray:
        size = min(per_group, len(pool))
        return rng.choice(pool, size=size, replace=False)

    pick_low  = _pick(pool_low)
    pick_mid  = _pick(pool_mid)
    pick_high = _pick(pool_high)
    pick_usr  = np.concatenate([pick_low, pick_mid, pick_high])
    print(f'Selected           : low={len(pick_low):,}  mid={len(pick_mid):,}  high={len(pick_high):,}')

    # Track segment membership for user_segments.pkl (Cell 9).
    segments = {
        'low':  [int(u) for u in pick_low],
        'mid':  [int(u) for u in pick_mid],
        'high': [int(u) for u in pick_high],
    }

    tst_int   = [None] * usrnum
    val_int   = [None] * usrnum
    test_rows: list[dict] = []
    val_rows:  list[dict] = []
    skipped = 0

    for u in pick_usr:
        merch = interaction[u]
        if merch is None or len(merch) < 3:
            skipped += 1
            continue

        # Flatten all (timestamp, merchant) events and sort chronologically
        pairs = sorted(
            (ts, mid)
            for mid, tss in merch.items()
            for ts in tss
        )

        tst_time, tst_mid = pairs[-1]

        # Find val: most recent event on a *different* merchant
        val_mid = val_time = None
        for ts, mid in reversed(pairs[:-1]):
            if mid != tst_mid:
                val_mid, val_time = mid, ts
                break
        if val_mid is None:
            skipped += 1
            continue

        # Remove only the held-out timestamp (not all visits to that merchant)
        tss = [t for t in interaction[u][tst_mid] if t != tst_time]
        if tss:
            interaction[u][tst_mid] = tss
        else:
            del interaction[u][tst_mid]

        tss = [t for t in interaction[u].get(val_mid, []) if t != val_time]
        if tss:
            interaction[u][val_mid] = tss
        elif val_mid in interaction[u]:
            del interaction[u][val_mid]

        # Post-deletion check: need at least 1 training interaction
        if sum(1 for tss in interaction[u].values() if tss) < 1:
            skipped += 1
            continue

        tst_int[u] = tst_mid
        val_int[u] = val_mid

        all_pos = set(merch.keys())
        # Independent negative samples for test and val
        test_negs = negSamp(all_pos, test_neg, merchantnum)
        val_negs  = negSamp(all_pos, test_neg, merchantnum)

        test_rows.append({
            'user_id': u + 1, 'merchant_id': tst_mid,
            'time': tst_time, 'neg_merchants': str(test_negs),
        })
        val_rows.append({
            'user_id': u + 1, 'merchant_id': val_mid,
            'time': val_time,  'neg_merchants': str(val_negs),
        })

    pd.DataFrame(test_rows).to_csv(OUT_DIR + 'test_yelp_merchant.csv',  sep='\t', index=False)
    pd.DataFrame(val_rows).to_csv( OUT_DIR + 'val_yelp_merchant.csv',   sep='\t', index=False)

    n_test = sum(1 for x in tst_int if x is not None)
    n_val  = sum(1 for x in val_int  if x is not None)
    per_seg = {k: sum(1 for u in v if tst_int[u] is not None) for k, v in segments.items()}
    print(f'Test users : {n_test:,}  (per segment: {per_seg})')
    print(f'Val  users : {n_val:,}')
    print(f'Skipped    : {skipped:,}')
    meta = {'thresholds': [float(t1), float(t2)],
            'criterion': 'total_event_count',
            'per_group_target': int(per_group),
            'per_segment_count': per_seg}
    return interaction, tst_int, val_int, segments, meta


interaction_train = copy.deepcopy(interaction)
interaction_train, tstInt, valInt, segments, segment_meta = split_test_val(
    interaction_train, usrnum, merchantnum
)

Tercile thresholds : low≤6, mid≤10, high>10
Pool sizes         : low=10,383  mid=8,048  high=8,237
Selected           : low=3,333  mid=3,333  high=3,333
Test users : 9,999  (per segment: {'low': 3333, 'mid': 3333, 'high': 3333})
Val  users : 9,999
Skipped    : 0


In [7]:
# ── Cell 7: Build graph matrices ────────────────────────────────────────────────

def trans(interaction: list, usrnum: int, merchantnum: int) -> csr_matrix:
    r, c, d = [], [], []
    for uid in range(usrnum):
        if not interaction[uid]:
            continue
        for mid, tss in interaction[uid].items():
            for _ in tss:
                r.append(uid); c.append(mid); d.append(1.0)
    mat = csr_matrix((d, (r, c)), shape=(usrnum, merchantnum))
    print(f'Global matrix: {mat.shape}, {(mat!=0).nnz:,} unique pairs, {len(d):,} total events')
    return mat


def trans_sub(
    interaction: list, usrnum: int, merchantnum: int,
    graph_num: int, minn: int, maxx: int,
) -> tuple:
    interval = (maxx - minn) / graph_num
    rcd       = [([],[],[]) for _ in range(graph_num)]
    seen      = [set() for _ in range(graph_num)]
    last_int: dict = {}

    for uid in range(usrnum):
        if not interaction[uid]:
            continue
        for mid, tss in interaction[uid].items():
            for ts in tss:
                t   = max(0, min(int((ts - minn) / interval), graph_num - 1))
                key = (uid, mid)
                if key not in seen[t]:
                    rcd[t][0].append(uid); rcd[t][1].append(mid); rcd[t][2].append(1.0)
                    seen[t].add(key)
                if key not in last_int or last_int[key] < t:
                    last_int[key] = t

    int_mats = []
    for t in range(graph_num):
        mat = csr_matrix((rcd[t][2], (rcd[t][0], rcd[t][1])), shape=(usrnum, merchantnum))
        int_mats.append(mat)
        print(f'  A_{t}: {mat.nnz:,} non-zeros')

    tr, tc, td = [], [], []
    for (u, i), t in last_int.items():
        tr.append(u); tc.append(i); td.append(t)
    time_mat = csr_matrix((td, (tr, tc)), shape=(usrnum, merchantnum))
    print(f'  timeMat: {time_mat.nnz:,} non-zeros')
    return int_mats, time_mat


global_mat = trans(interaction_train, usrnum, merchantnum)
print(f'\nBuilding T={GRAPH_NUM} time-interval sub-graphs...')
subMat, timeMat = trans_sub(interaction_train, usrnum, merchantnum, GRAPH_NUM, minn, maxx)

Global matrix: (1210, 17377), 136,415 unique pairs, 2,747,386 total events

Building T=8 time-interval sub-graphs...
  A_0: 56,658 non-zeros
  A_1: 56,887 non-zeros
  A_2: 57,677 non-zeros
  A_3: 57,678 non-zeros
  A_4: 56,278 non-zeros
  A_5: 56,925 non-zeros
  A_6: 57,726 non-zeros
  A_7: 57,103 non-zeros
  timeMat: 136,415 non-zeros


In [8]:
# ── Cell 8: Build sequence + save all pickles ───────────────────────────────────

sequence: list = []
empty_count = 0
for uid in range(usrnum):
    if not interaction_train[uid]:
        sequence.append([0])
        empty_count += 1
        continue
    pairs = sorted((ts, mid) for mid, tss in interaction_train[uid].items() for ts in tss)
    sequence.append([mid for _, mid in pairs] if pairs else [0])
    if not pairs:
        empty_count += 1

# Consistency assertion
seq_events = sum(len(s) for s in sequence) - empty_count
mat_events = int(global_mat.sum())
assert seq_events == mat_events, (
    f'MISMATCH: sequence has {seq_events:,} events but global_mat has {mat_events:,}'
)
print(f'Sequence events : {seq_events:,}')
print(f'Global mat sum  : {mat_events:,}')
print('Sequence-matrix consistency check PASSED.')

# Save core pickles
with open(OUT_DIR + 'trn_mat_time', 'wb') as f: pickle.dump([global_mat, subMat, timeMat], f)
with open(OUT_DIR + 'tst_int',      'wb') as f: pickle.dump(tstInt, f)
with open(OUT_DIR + 'val_int',      'wb') as f: pickle.dump(valInt, f)
with open(OUT_DIR + 'sequence',     'wb') as f: pickle.dump(sequence, f)
print('Saved: trn_mat_time, tst_int, val_int, sequence')

# Balanced low/mid/high segment membership for segment-wise evaluation.
with open(OUT_DIR + 'user_segments.pkl', 'wb') as f:
    pickle.dump({'segments': segments, 'meta': segment_meta}, f)
print(f"Saved: user_segments.pkl (low={len(segments['low']):,} | "
      f"mid={len(segments['mid']):,} | high={len(segments['high']):,})")

# Preserve the 2-year window bounds for feature_extraction (used to derive
# cutoff without a second CSV pass).
import json as _json
with open(OUT_DIR + 'window_config.json', 'w') as f:
    _json.dump({'cutoff_ts':    int(minn),
                'max_ts':       int(maxx),
                'last_n_years': int(LAST_N_YEARS),
                'graph_num':    int(GRAPH_NUM)}, f, indent=2)
print(f'Saved: window_config.json (cutoff={minn}, max={maxx}, T={GRAPH_NUM})')

# test_dict / val_dict
for split, csv_name in [('test', 'test_finance_merchant.csv'), ('val', 'val_finance_merchant.csv')]:
    df_split = pd.read_csv(OUT_DIR + csv_name, sep='\t')
    d: dict = {}
    for _, row in df_split.iterrows():
        uid  = int(row['user_id']) - 1
        negs = ast.literal_eval(str(row['neg_merchants']))
        d[uid] = negs
    with open(OUT_DIR + f'{split}_dict', 'wb') as f:
        pickle.dump(d, f)
    print(f'Saved: {split}_dict ({len(d):,} users, 0-indexed)')

# Train CSV
train_rows: list = []
for uid in range(usrnum):
    if not interaction_train[uid]:
        continue
    for mid, tss in interaction_train[uid].items():
        for ts in tss:
            train_rows.append({'user_id': uid + 1, 'merchant_id': mid, 'time': ts})
df_train = pd.DataFrame(train_rows)
df_train.to_csv(OUT_DIR + 'train_finance_merchant.csv', sep='\t', index=False)
print(f'Saved: train_finance_merchant.csv ({len(df_train):,} rows)')

# Edge weights
with open(OUT_DIR + 'edge_weights.pkl', 'wb') as f:
    pickle.dump(edge_weights, f)
print(f'Saved: edge_weights.pkl ({len(edge_weights):,} pairs)')

Sequence events : 2,747,386
Global mat sum  : 2,747,386
Sequence-matrix consistency check PASSED.
Saved: trn_mat_time, tst_int, val_int, sequence
Saved: user_segments.pkl (low=404 | mid=403 | high=403)
Saved: window_config.json (cutoff=1509494340, max=1572566340, T=8)
Saved: test_dict (1,210 users, 0-indexed)
Saved: val_dict (1,210 users, 0-indexed)
Saved: train_finance_merchant.csv (2,747,386 rows)
Saved: edge_weights.pkl (136,459 pairs)


In [9]:
# ── Cell 9: Verification & summary ─────────────────────────────────────────────

df_test = pd.read_csv(OUT_DIR + 'test_finance_merchant.csv', sep='\t')
df_val  = pd.read_csv(OUT_DIR + 'val_finance_merchant.csv',  sep='\t')

train_pairs = set(zip(df_train['user_id'], df_train['merchant_id']))
test_pairs  = set(zip(df_test['user_id'],  df_test['merchant_id']))
val_pairs   = set(zip(df_val['user_id'],   df_val['merchant_id']))

overlap_tr_ts = train_pairs & test_pairs
overlap_tr_vl = train_pairs & val_pairs
overlap_ts_vl = test_pairs  & val_pairs

print(f'Train-Test overlap : {len(overlap_tr_ts):,} pairs')
print(f'Train-Val  overlap : {len(overlap_tr_vl):,} pairs')
print(f'Test-Val   overlap : {len(overlap_ts_vl):,} pairs')
if overlap_tr_ts:
    print('  (expected: earlier visits to same merchant remain in train)')

assert len(overlap_ts_vl) == 0, f'FAIL: Test-Val overlap = {len(overlap_ts_vl)} pairs'
print('Test-Val disjointness check PASSED.')

# File inventory
required_files = [
    'trn_mat_time', 'tst_int', 'val_int', 'sequence',
    'test_dict', 'val_dict', 'user2id.pkl', 'merchant2id.pkl',
    'edge_weights.pkl',
    'train_finance_merchant.csv', 'test_finance_merchant.csv', 'val_finance_merchant.csv',
]
print()
print(f'{"File":<35} {"Size"}')
print('-' * 47)
for fname in required_files:
    path   = OUT_DIR + fname
    size   = os.path.getsize(path) if os.path.isfile(path) else None
    status = f'{size / 1024:.0f} KB' if size else 'MISSING'
    print(f'  {fname:<33} {status}')

print()
print('=' * 60)
print('Preprocessing Complete — finance-merchant')
print('=' * 60)
print(f'Users                : {usrnum:>10,}')
print(f'Merchants            : {merchantnum:>10,}')
print(f'Train events         : {len(df_train):>10,}')
print(f'Unique train pairs   : {(global_mat != 0).nnz:>10,}')
print(f'Test users           : {len(df_test):>10,}')
print(f'Val  users           : {len(df_val):>10,}')
print(f'Negatives per user   : {TEST_NEG:>10,}')
print(f'Time intervals (T)   : {GRAPH_NUM:>10}')
print(f'Output dir           : {OUT_DIR}')

Train-Test overlap : 1,188 pairs
Train-Val  overlap : 1,188 pairs
Test-Val   overlap : 0 pairs
  (expected: earlier visits to same merchant remain in train)
Test-Val disjointness check PASSED.

File                                Size
-----------------------------------------------
  trn_mat_time                      8068 KB
  tst_int                           4 KB
  val_int                           4 KB
  sequence                          8045 KB
  test_dict                         3533 KB
  val_dict                          3532 KB
  user2id.pkl                       11 KB
  merchant2id.pkl                   185 KB
  edge_weights.pkl                  2237 KB
  train_finance_merchant.csv        57146 KB
  test_finance_merchant.csv         7535 KB
  val_finance_merchant.csv          7535 KB

Preprocessing Complete — finance-merchant
Users                :      1,210
Merchants            :     17,377
Train events         :  2,747,386
Unique train pairs   :    136,415
Test users        